# Prompt Eval Results Viewer

Load and compare `EvaluationReport` JSON files produced by `prompt_eval`.

Usage:
- Set `RESULTS_DIR` to the directory containing result JSON files
- Run all cells to load and display results
- Use the comparison section to diff two runs side-by-side

In [ ]:
from pathlib import Path
import json
import pandas as pd
from llm_metadata.groundtruth_eval import EvaluationReport

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# List available result files
result_files = sorted(RESULTS_DIR.glob("*.json"))
print(f"Found {len(result_files)} result files:")
for f in result_files:
    print(f"  {f.name}")

## Load a Single Run

In [ ]:
# Load a single run — edit path as needed
RUN_PATH = RESULTS_DIR / "baseline_abstract.json"

if RUN_PATH.exists():
    report = EvaluationReport.load(RUN_PATH)
    
    # Show run metadata
    with open(RUN_PATH) as f:
        meta = json.load(f)
    
    print("Run metadata:")
    for k, v in meta.items():
        if k not in ("field_metrics", "field_results", "config"):
            print(f"  {k}: {v}")
else:
    print(f"File not found: {RUN_PATH}")
    print("Run prompt_eval first to generate results.")
    report = None

## Per-Field Metrics Table

In [ ]:
if report is not None:
    metrics_df = report.metrics_to_pandas()
    metrics_df = metrics_df.sort_values("f1", ascending=False)
    
    # Format for display
    display_df = metrics_df[["field", "n", "tp", "fp", "fn", "precision", "recall", "f1"]].copy()
    for col in ["precision", "recall", "f1"]:
        display_df[col] = display_df[col].map(lambda x: f"{x:.3f}" if x is not None else "-")
    
    display(display_df.reset_index(drop=True))

## Mismatch Explorer

In [ ]:
if report is not None:
    FIELD = "species"  # Change to inspect different fields
    
    errors = report.errors_for_field(FIELD)
    print(f"Mismatches for '{FIELD}': {len(errors)} / {report.field_metrics[FIELD].n if FIELD in report.field_metrics else '?'}")
    
    for r in errors[:10]:  # Show first 10 mismatches
        print(f"\n  record_id: {r.record_id}")
        print(f"  true:  {r.true_value}")
        print(f"  pred:  {r.pred_value}")
        print(f"  tp={r.tp}, fp={r.fp}, fn={r.fn}")

## Side-by-Side Comparison of Two Runs

In [ ]:
RUN_A = RESULTS_DIR / "baseline_abstract.json"
RUN_B = RESULTS_DIR / "run2.json"  # second run to compare

if RUN_A.exists() and RUN_B.exists():
    report_a = EvaluationReport.load(RUN_A)
    report_b = EvaluationReport.load(RUN_B)
    
    metrics_a = report_a.metrics_to_pandas().set_index("field")
    metrics_b = report_b.metrics_to_pandas().set_index("field")
    
    common_fields = sorted(set(metrics_a.index) & set(metrics_b.index))
    
    rows = []
    for field in common_fields:
        f1_a = metrics_a.loc[field, "f1"]
        f1_b = metrics_b.loc[field, "f1"]
        delta = (f1_b - f1_a) if (f1_a is not None and f1_b is not None) else None
        rows.append({
            "field": field,
            "f1_A": f"{f1_a:.3f}" if f1_a is not None else "-",
            "f1_B": f"{f1_b:.3f}" if f1_b is not None else "-",
            "delta": f"{delta:+.3f}" if delta is not None else "-",
        })
    
    comparison_df = pd.DataFrame(rows)
    display(comparison_df)
else:
    print("One or both result files not found. Run prompt_eval to generate results.")